# 5. Segment and quantify

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SMLCI/acia-core/blob/main/docs/tutorials/05_segment_and_quantify.ipynb)

This is the one that pays for the previous four. We take the raw time-lapse,
segment every cell with a deep-learning model, measure them in physical units,
throw out the artefacts, and end with a **growth rate** — the kind of number that
goes in a figure.

Notably, none of this needs tracking. {func}`~acia.analysis.extract_growth`
aggregates cell area per timepoint, so you get a population growth rate from
segmentation alone. Tracking is what you add when you want *per-lineage*
answers.

:::{tip}
This notebook uses a GPU when one is available and falls back to CPU when it is
not — it just processes fewer frames. On Colab, pick a GPU runtime via
**Runtime → Change runtime type** for the full-resolution experience.
:::

In [ ]:
# On Colab (or any fresh environment) this installs acia.
# Locally, if you already have acia installed, it is a no-op.
try:
    import acia  # noqa: F401
except ImportError:
    %pip install -q acia

## Install a segmentation backend

We use **Cellpose-SAM**. `acia` supports several backends, but they pin
conflicting versions of `cellpose`, `torch` and `numpy`, so **exactly one can be
installed per environment** — see {doc}`/installation`. Swapping backends is a
one-line change in the cell further down; swapping environments is the price.

In [ ]:
try:
    import cellpose  # noqa: F401
except ImportError:
    %pip install -q "cellpose>=4"

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_URL = "https://data.celltrackingchallenge.net/training-datasets/DIC-C2DH-HeLa.zip"
DATA_DIR = Path("data")
SEQUENCE = DATA_DIR / "DIC-C2DH-HeLa" / "01"

if not SEQUENCE.exists():
    DATA_DIR.mkdir(exist_ok=True)
    archive = DATA_DIR / "DIC-C2DH-HeLa.zip"
    print("downloading ~42 MB ...")
    urlretrieve(DATA_URL, archive)
    with ZipFile(archive) as zf:
        zf.extractall(DATA_DIR)
    archive.unlink()

print(SEQUENCE, "->", len(list(SEQUENCE.glob("*.tif"))), "frames")

## Pick a working size

Segmentation cost scales with frames and pixels, so we choose how much to do
based on what hardware we actually have. This is the subsampling habit from
[tutorial 2](02_the_sequence_model.ipynb) doing real work.

In [ ]:
import torch

# CUDA on Colab and most workstations; MPS on Apple silicon.
ACCELERATED = torch.cuda.is_available() or torch.backends.mps.is_available()
STEP = 4 if ACCELERATED else 20  # every 4th frame accelerated, every 20th on CPU

print(
    "accelerator:",
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "none (CPU)",
)
if not ACCELERATED:
    print("Running on CPU -- using fewer frames. This still works, just coarser.")

In [ ]:
from acia import ureg
from acia.segm.open import open_sequence

full = open_sequence(SEQUENCE).position(0)
full = full.with_pixel_size(0.19 * ureg.micrometer).with_frame_interval(
    10 * ureg.minute
)

src = full[::STEP]

print(f"{len(full)} frames -> {len(src)} frames")
print("time span:", src.timepoints[-1].to("hour"))

## Segment

A segmenter is a callable: hand it a source, get back an
{class}`~acia.base.Overlay` of detections. The model is loaded lazily on first
use and, by default, released afterwards so the GPU memory goes back to the
system.

In [ ]:
from acia.segm.processor.cellpose_sam import CellposeSAMSegmenter

segmenter = CellposeSAMSegmenter()

overlay = segmenter(src)

print(len(overlay), "detections across", overlay.numFrames(), "frames")

Each detection knows its frame, its id and its area in **pixels** — the raw
geometric measurement, before any calibration is applied.

One thing a segmenter does *not* do is attach a time model: `contour.time` is
`None` until the overlay is told what the frames mean. (Trackers do this for you;
a bare segmentation does not.) Attaching it is one call, and worth doing because
it makes every detection self-describing.

In [ ]:
first = next(iter(overlay))
print(
    "before:", first.frame, first.id, round(first.area, 1), "px^2, time =", first.time
)

overlay = overlay.with_timepoints(src.timepoints)

first = next(iter(overlay))
print(
    "after :", first.frame, first.id, round(first.area, 1), "px^2, time =", first.time
)

## See what the model did

Never trust a segmentation you have not looked at.
{func}`~acia.viz.render_segmentation_mask` paints the masks over the image and,
as always, returns a source — so it composes with the annotation and video
helpers from [tutorial 3](03_look_at_your_data.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from acia.viz import render_segmentation_mask

painted = render_segmentation_mask(src.to_rgb(), overlay, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, idx in zip(axes, [0, len(src) // 2, len(src) - 1], strict=True):
    ax.imshow(np.asarray(painted[idx].raw))
    ax.set_title(f"{src.timepoints[idx].to('hour'):~.1f}")
    ax.axis("off")
fig.suptitle("Cellpose-SAM segmentation")
fig.tight_layout()
plt.show()

In [ ]:
from IPython.display import Video

from acia.viz import render_video

render_video(painted, "segmented.mp4", framerate=4)
Video("segmented.mp4", embed=True, width=420)

## Measure

{class}`~acia.analysis.ExtractorExecutor` turns the overlay into a tidy,
id-indexed DataFrame — one row per detection, one column per property. Because
the source is calibrated, areas come out in µm² and times in hours without any
further configuration.

In [ ]:
from acia.analysis import (
    AreaEx,
    BoundaryClosenessEx,
    CircularityEx,
    ExtractorExecutor,
    FrameEx,
    PerimeterEx,
    TimeEx,
)

properties = ExtractorExecutor().execute(
    overlay,
    src,
    extractors=[
        FrameEx(),
        TimeEx(),
        AreaEx(),
        PerimeterEx(),  # CircularityEx is derived from area and perimeter,
        CircularityEx(),  # so PerimeterEx must come before it
        BoundaryClosenessEx(),
    ],
)

print(properties.head())
print()
print("units:", properties.attrs["units"])

## Throw out the artefacts

Real segmentations contain debris and merged blobs.
{func}`~acia.segm.filter.apply_cell_filters` takes the measured table and a list
of filters, with thresholds in physical units — which is the whole reason we
bothered with calibration.

In [ ]:
from acia.segm.filter import AreaFilter, apply_cell_filters

filters = [AreaFilter(vmin=50 * ureg.micrometer**2, vmax=2000 * ureg.micrometer**2)]

filtered_overlay = apply_cell_filters(overlay, filters, properties=properties)

print(
    f"{len(overlay)} detections -> {len(filtered_overlay)} kept "
    f"({len(overlay) - len(filtered_overlay)} removed)"
)

{func}`~acia.analysis.properties.plot_property_histograms` shows the before and
after distributions together, so you can see what a threshold actually did rather
than guessing.

In [ ]:
from acia.analysis.properties import plot_property_histograms

properties_after = ExtractorExecutor().execute(
    filtered_overlay,
    src,
    extractors=[FrameEx(), TimeEx(), AreaEx(), PerimeterEx(), CircularityEx()],
)

plot_property_histograms(
    properties,
    ["area", "circularity"],
    df_after=properties_after,
    show_removed=True,
)
plt.show()

## The growth rate

{func}`~acia.analysis.extract_growth` does the last step in one call: it
aggregates total cell area per timepoint, fits an exponential model, and returns
the table, the fit result and a figure.

In [ ]:
from acia.analysis import extract_growth

table, result, figure = extract_growth(filtered_overlay, src, time_unit="hour")

print(table.head())
print()
print("growth rate  :", result.growth_rate)
print("doubling time:", result.doubling_time)
print("R^2          :", round(result.r_squared, 4))
plt.show()

## A filter that would be wrong here

`acia` also ships {class}`~acia.segm.filter.BoundaryClosenessFilter`, which drops
cells touching the edge of the field of view. That is usually a *good* idea — a
cell half outside the image has a meaningless area.

On this dataset it is a trap, and it is worth seeing why, because the failure is
silent: you still get a growth rate, it is just wrong.

In [ ]:
from acia.segm.filter import BoundaryClosenessFilter

edge_filtered = apply_cell_filters(
    overlay,
    [*filters, BoundaryClosenessFilter(min_distance=2 * ureg.micrometer)],
    properties=properties,
)
_, edge_result, _ = extract_growth(edge_filtered, src, time_unit="hour")

print(f"kept {len(edge_filtered)}/{len(overlay)} detections")
print("doubling time:", edge_result.doubling_time)
print("R^2          :", round(edge_result.r_squared, 4))

Half the population is gone, and the fit collapses.

The reason is that these are large adherent HeLa cells covering a small field of
view, so *which* cells touch the border is not random noise — it is correlated
with the very thing being measured. As the colony grows, more of it reaches the
edge, so the surviving cells' total area stays roughly flat and the growth signal
is filtered away along with the artefacts.

The lesson generalises: a filter is only safe when what it removes is
**independent of the quantity you are measuring**. `BoundaryClosenessFilter`
earns its keep on cells confined in a microfluidic chamber with free space around
them; here it does not. Always compare the before/after distributions — and the
resulting fit — rather than applying a filter because it sounds prudent.

:::{note}
Treat the exact number with the scepticism it deserves: this is a short,
heavily subsampled clip of one field of view, segmented with a general-purpose
model and no parameter tuning. What matters here is that the *pipeline* is
complete and every quantity carries its unit — swapping in your own data, a
tuned model and the full frame count is a matter of changing the parameters, not
the code.
:::

## Where to go next

You now have the full loop: **open → slice → visualize → segment → measure →
filter → quantify.**

* **Tracking and lineages.** Add a tracker
  ({class}`~acia.tracking.processor.trackastra.TrackastraTracker`,
  `UltrackTracker`, `LaptrackTracker`, `PyUATTracker`) to follow individual cells
  through divisions, then plot lineage trees and per-cell doubling times.
* **A different backend.** Replace `CellposeSAMSegmenter` with
  `OmniposeSegmenter` (excellent on bacteria), `CellposeSegmenter`, `CPNSegmenter`
  or `YOLOSegmenter` — same call signature, different environment.
* **Scale it up.** {func}`acia.analysis.scale` runs this notebook once per
  sequence across hundreds of positions; see {doc}`/guide/scaling`.
* **Real experiments.** The
  [acia-workflows](https://github.com/JuBiotech/acia-workflows)
  collection has complete published analyses — growth-rate quantification,
  fluorescence co-culture, single-cell oxygen response — built on exactly these
  pieces.